# Phase 6 — Explainability: SHAP Feature Attributions on Primary Classifier

**Status:** ✅ Completed (pipeline v1.0)  
**Authoritative Script:** `ml_service/src/shap_explain.py`  
**Primary Model Analyzed:** `models/xgb_baseline_category_v1.pkl` (XGBoost Native Booster)  
**Technical Report:** `artifacts/shap_summary_v1.md`  
**Data Artifacts:** `artifacts/shap_values_v1.npz` & `artifacts/shap_summary_v1.json`  

---

## Executive Summary & Research Methodology
Phase 6 implements SHAP (SHapley Additive exPlanations) for the primary XGBoost model selected in Phase 5.
Key methodological controls enforced:
- **Explainer:** `shap.TreeExplainer` with `feature_perturbation='tree_path_dependent'` (no background reference set required; exact tree conditioning).
- **Sampling Protocol:** Per-class capped sampling (up to 150 rows per category) across the 504,160-row test split, yielding **1,793 balanced flows** and preventing BENIGN traffic from drowning rare attack categories.
- **Memory Safety:** Full SHAP array consumes **7.8 MB** (float32), peaking at 587.1 MB RSS — strictly adhering to the 8 GB machine budget.
- **Mathematical Rigor:** Formal additivity check passed (`base_value + sum(SHAP) == margin`) with max discrepancy **`7.63e-06`**.
- **Global Ranking Metric:** **Macro-averaged mean |SHAP|** across all 15 classes to eliminate sample-frequency bias.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
try:
    from IPython.display import display
except ImportError:
    display = print

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

# Load authoritative SHAP summary JSON
with open(Paths.ARTIFACTS_DIR / 'shap_summary_v1.json', 'r', encoding='utf-8') as f:
    shap_summary = json.load(f)

# Load compressed SHAP values
shap_npz = np.load(Paths.ARTIFACTS_DIR / 'shap_values_v1.npz')
shap_values = shap_npz['shap_values']
expected_values = shap_npz['expected_values']
y_sample = shap_npz['y_sample']
y_pred = shap_npz['y_pred']
class_names = list(shap_npz['class_names'])
feature_names = list(shap_npz['feature_names'])

meta = shap_summary['meta']
print('Loaded Phase 6 SHAP Artifacts Successfully.')
print(f"Sample Count: {meta['sample_count']} flows | Features: {meta['feature_count']} | Classes: {meta['class_count']}")
print(f"Additivity Check: {'PASSED' if meta['additivity']['passed'] else 'FAILED'} (Max Diff: {meta['additivity']['max_absolute_difference']:.2e})")
print(f"Execution Time: {meta['computation_time_seconds']:.2f} s | SHAP Array Size: {shap_values.nbytes / (1024*1024):.1f} MB")

## 1. Global Feature Importance (Macro-Averaged)
Because our sample uses a per-class cap rather than proportional sampling, computing the global mean |SHAP| by simple pooling would overstate rare classes. We therefore compute the **macro-averaged mean |SHAP|**: the mean per class, then averaged across all 15 classes.

In [ ]:
global_rankings = shap_summary['global_macro_rankings']
df_global = pd.DataFrame(global_rankings[:15])
df_global.columns = ['Rank', 'Feature', 'Macro Mean |SHAP|']

# Add functional group
def categorize(feat):
    if 'IAT' in feat: return 'Timing / IAT'
    if 'Flag' in feat: return 'TCP Flags'
    if any(k in feat for k in ['Length', 'Bytes', 'Packets']): return 'Packet Sizing / Volume'
    return 'Flow Geometry / Window'

df_global['Functional Group'] = df_global['Feature'].apply(categorize)
df_display = df_global.copy()
df_display['Macro Mean |SHAP|'] = df_display['Macro Mean |SHAP|'].apply(lambda x: f'{x:.4f}')
display(df_display)

In [ ]:
# Display Global Macro-Averaged Bar Plot
plot_path = Paths.ARTIFACTS_DIR / 'shap_plots' / 'shap_global_macro_bar.png'
if plot_path.exists():
    img = mpimg.imread(str(plot_path))
    plt.figure(figsize=(12, 9), dpi=150)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Global Feature Attributions (XGBoost)', fontsize=12, fontweight='bold')
    plt.show()

## 2. Multi-Class Attribution Profiles across Threat Families
This heatmap illustrates the average feature attributions for key network traffic families across the top global features:

In [ ]:
plot_path = Paths.ARTIFACTS_DIR / 'shap_plots' / 'shap_core_classes_heatmap.png'
if plot_path.exists():
    img = mpimg.imread(str(plot_path))
    plt.figure(figsize=(12, 7), dpi=150)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 3. Prior Domain Hypotheses vs. Empirical SHAP Findings
Per academic research integrity standards, domain expectations were formally pre-recorded as hypotheses. We evaluate whether the trained model genuinely relied on these mechanisms or identified distinct discriminatory signals:

> **Crucial Domain Note on PortScan:** In Phase 2, `Destination Port` was explicitly isolated as an identifier (excluded from ML features). Consequently, PortScan cannot be detected via port diversity across flows. Instead, the model relies on single-flow geometry (`min_seg_size_forward`, packet length distributions, and packet volume).

In [ ]:
hyp_eval = shap_summary['hypotheses_evaluation']
hyp_rows = []
for category, data in hyp_eval.items():
    hyp_rows.append({
        'Threat Category': category,
        'Hypothesis Supported': 'Yes' if data['hypothesis_supported'] else 'No',
        'Prior Hypothesis': data['prior_hypothesis'],
        'Empirical Top Features': ', '.join(data['empirical_top_5'][:3]),
        'Domain Finding': data['scientific_note']
    })
df_hyp = pd.DataFrame(hyp_rows)
display(df_hyp)

## 4. Local Explanations: Representative True Positives & Error Attribution
Here we visualize waterfall plots for high-confidence correct classifications, as well as an error attribution case where the model misclassified a sample.

In [ ]:
# True Positives
fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=150)
tp_files = [
    ('DoS Hulk TP', 'waterfall_dos_hulk_tp.png'),
    ('PortScan TP', 'waterfall_portscan_tp.png'),
    ('FTP-Patator TP', 'waterfall_ftp_patator_tp.png'),
    ('DDoS TP', 'waterfall_ddos_tp.png'),
]

for ax, (title, filename) in zip(axes.flatten(), tp_files):
    p = Paths.ARTIFACTS_DIR / 'shap_plots' / filename
    if p.exists():
        img = mpimg.imread(str(p))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Error Attribution Case
err_path = Paths.ARTIFACTS_DIR / 'shap_plots' / 'waterfall_misclassified_case.png'
if err_path.exists():
    img = mpimg.imread(str(err_path))
    plt.figure(figsize=(11, 7), dpi=150)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Misclassification Attribution Analysis', fontsize=12, fontweight='bold')
    plt.show()

## 5. Phase 11 Decision-Support Handoff: `top_k_contributions` Utility
To support grounded LLM cyber defense guidance in Phase 11, we verify our modular utility which extracts the top positive and negative contributing features for any individual flow prediction:

In [ ]:
from ml_service.src.shap_explain import get_top_k_contributions

# Test on a representative DoS Hulk sample (Index 0)
sample_idx = 0
predicted_class = int(y_pred[sample_idx])

contributions = get_top_k_contributions(
    shap_values_row=shap_values[sample_idx],
    feature_names=feature_names,
    class_idx=predicted_class,
    k=3
)

print(f"Flow Sample #{sample_idx} - Ground Truth: {class_names[y_sample[sample_idx]]} | Predicted: {class_names[predicted_class]}")
print("\nStructured JSON Evidence Payload for Gemini Analyst Prompt:")
print(json.dumps(contributions, indent=2))